In [11]:
import pandas as pd

In [12]:
bd = pd.read_csv("cha_bd.bed", sep="\t",header=None)
bd.columns = ["chrom", "start", "end"]
bd

,chrom,start,end
0,chr1,1980000,2020000
1,chr1,2860000,2900000
2,chr1,3380000,3420000
3,chr1,3580000,3620000
4,chr1,3580000,3620000
...,...,...,...
7301,chrX,125020000,125060000
7302,chrX,125020000,125060000
7303,chrX,125220000,125260000
7304,chrX,125220000,125260000


In [13]:
wt = pd.read_csv("../../../IS/WT_is.tsv", sep="\t")
wt_final = wt[wt["is_bad_bin"]==False]

In [14]:
import bioframe as bf
import bioframe.vis

olwt = bf.overlap(bd, wt_final, how="inner", suffixes=('_1','_2'))
olwt_final = olwt.groupby(["chrom_1","start_1","end_1"])["log2_insulation_score_1000000_2"].apply(list).reset_index(name="wt_is")

In [17]:
pi = pd.read_csv("../../../IS/PI_is.tsv", sep="\t")
pi_final = pi[pi["is_bad_bin"]==False]

In [18]:
olpi = bf.overlap(bd, pi_final, how="inner", suffixes=('_1','_2'))
olpi_final = olpi.groupby(["chrom_1","start_1","end_1"])["log2_insulation_score_1000000_2"].apply(list).reset_index(name="pi_is")

In [19]:
out = pd.merge(olwt_final, olpi_final)
out

,chrom_1,start_1,end_1,wt_is,pi_is
0,chr1,1940000,1980000,"[-0.0280213861779435, -0.0682799402049146, -0....","[-0.2823163326328322, -0.3329704686296486, -0...."
1,chr1,1980000,2020000,"[-0.1868910769014775, -0.2173613135743544, -0....","[-0.4518239339014759, -0.4828280630465671, -0...."
2,chr1,2660000,2700000,"[-0.7281472907532525, -0.7249848340579974, -0....","[-0.8220003528534334, -0.8198841559874085, -0...."
3,chr1,2860000,2900000,"[-0.5317124049569716, -0.5386853922210064, -0....","[-0.5465335534724333, -0.559866073161156, -0.6..."
4,chr1,3380000,3420000,"[-0.3253091474415295, -0.3319094944138893, -0....","[-0.3458920782419459, -0.3530509467360012, -0...."
...,...,...,...,...,...
3893,chrX,124980000,125020000,"[-0.3373980151814856, -0.349682100810083, -0.3...","[-0.3519176547527021, -0.3628814976025473, -0...."
3894,chrX,125020000,125060000,"[-0.4499179567523094, -0.4941928226690172, -0....","[-0.4559505800529054, -0.5098045736177517, -0...."
3895,chrX,125220000,125260000,"[-0.0734559724053479, -0.0590544077899204, -0....","[-0.0155680811668754, 0.0029913430367695, 0.01..."
3896,chrX,125540000,125580000,"[0.6469492579166907, 0.6787590411108289, 0.694...","[0.8490507758356127, 0.8692743024091807, 0.897..."


In [20]:
def c_p():
    from scipy.stats import ttest_ind
    p = []
    for _, row in out.iterrows():
        stat,pvalue = ttest_ind(row['wt_is'], row['pi_is'])
        p.append(pvalue)
    return p

In [21]:
def Average(lst):
    return sum(lst) / len(lst)

def c_fc():
    from scipy.stats import ttest_ind
    fc = []
    for _, row in out.iterrows():
        f = Average(row['wt_is'])/Average(row['pi_is'])
        fc.append(f)
    return fc

In [22]:
p=c_p()
out["p-value"] = p
fc=c_fc()
out["fc"] = fc
out

/public/home/luzhang/miniconda3/envs/pic/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3757: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/public/home/luzhang/miniconda3/envs/pic/lib/python3.9/site-packages/numpy/core/_methods.py:256: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


,chrom_1,start_1,end_1,wt_is,pi_is,p-value,fc
0,chr1,1940000,1980000,"[-0.0280213861779435, -0.0682799402049146, -0....","[-0.2823163326328322, -0.3329704686296486, -0....",5.469984e-04,0.254098
1,chr1,1980000,2020000,"[-0.1868910769014775, -0.2173613135743544, -0....","[-0.4518239339014759, -0.4828280630465671, -0....",1.294241e-05,0.462995
2,chr1,2660000,2700000,"[-0.7281472907532525, -0.7249848340579974, -0....","[-0.8220003528534334, -0.8198841559874085, -0....",5.980812e-15,0.888948
3,chr1,2860000,2900000,"[-0.5317124049569716, -0.5386853922210064, -0....","[-0.5465335534724333, -0.559866073161156, -0.6...",1.627035e-01,0.964141
4,chr1,3380000,3420000,"[-0.3253091474415295, -0.3319094944138893, -0....","[-0.3458920782419459, -0.3530509467360012, -0....",7.202957e-05,0.939019
...,...,...,...,...,...,...,...
3893,chrX,124980000,125020000,"[-0.3373980151814856, -0.349682100810083, -0.3...","[-0.3519176547527021, -0.3628814976025473, -0....",3.614205e-01,0.967172
3894,chrX,125020000,125060000,"[-0.4499179567523094, -0.4941928226690172, -0....","[-0.4559505800529054, -0.5098045736177517, -0....",4.951747e-01,0.980772
3895,chrX,125220000,125260000,"[-0.0734559724053479, -0.0590544077899204, -0....","[-0.0155680811668754, 0.0029913430367695, 0.01...",6.272537e-07,-6.787357
3896,chrX,125540000,125580000,"[0.6469492579166907, 0.6787590411108289, 0.694...","[0.8490507758356127, 0.8692743024091807, 0.897...",7.004410e-10,0.772865


In [28]:
bd_change = out[(out["p-value"]<0.05)&(out["fc"]>1)]
bd_change.to_csv("bd_change.bed", sep="\t", index=False)